# Evaluation Notebook

This notebook evaluates tracking performance:
- Computing CTC metrics
- Analyzing tracking accuracy
- Comparing different segmentation/tracking methods
- Visualizing error patterns

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging
from collections import defaultdict

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

plt.rcParams['figure.figsize'] = (14, 8)
np.set_printoptions(precision=3, suppress=True)

VOXEL_SIZE_UM = (1.625, 0.40625, 0.40625)

print("=" * 60)
print("SETUP INSTRUCTIONS")
print("=" * 60)
print("\nTo run this notebook, execute from project root:")
print("\n  pip install -e .")
print("  pip install zarr")
print("\nThen restart the Jupyter kernel.")
print("=" * 60)

# Try importing modules
try:
    from biohub_tracking.data.zarr_loader import iter_frames
    from biohub_tracking.segmentation.segmenter import CellSegmenter
    from biohub_tracking.tracking.linker import HungarianLinker
    from biohub_tracking.evaluation.metrics import compute_ctc_metrics
    MODULES_AVAILABLE = True
    print("✓ biohub_tracking modules available")
except ImportError as e:
    MODULES_AVAILABLE = False
    print(f"✗ biohub_tracking not available - {e}")

## 1. Run Full Tracking Pipeline

In [ ]:
# Find sample data
data_dir = Path("../data")
sample_path = None

if (data_dir / "train").exists():
    samples = sorted((data_dir / "train").glob("*.zarr"))
    if samples:
        sample_path = samples[0]
elif (data_dir / "test").exists():
    samples = sorted((data_dir / "test").glob("*.zarr"))
    if samples:
        sample_path = samples[0]

if sample_path:
    print(f"Processing sample: {sample_path.name}")
    
    # Initialize components
    segmenter = CellSegmenter(
        method="blob",
        min_size=30,
        anisotropy=VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1],
        voxel_size_um=VOXEL_SIZE_UM
    )
    
    linker = HungarianLinker(
        max_distance=7.0,
        use_volume_cost=False
    )
    
    # Run pipeline
    all_cells = {}
    frame_count = 0
    max_frames = 10
    
    for frame_index, image in iter_frames(sample_path):
        if frame_count >= max_frames:
            break
            
        labels, cells = segmenter.segment_frame(image, frame_index=frame_index)
        all_cells[frame_index] = cells
        
        if frame_count % 2 == 0:
            print(f"Frame {frame_index}: {len(cells)} cells")
        frame_count += 1
    
    print(f"\nProcessed {frame_count} frames")
else:
    print("No sample data found")
    all_cells = {}

## 2. Compute Tracking Metrics

In [ ]:
if all_cells and len(all_cells) > 1:
    # Perform linking
    links = []
    linked_ids = {}
    
    sorted_frames = sorted(all_cells.keys())
    for i in range(len(sorted_frames)-1):
        frame1 = sorted_frames[i]
        frame2 = sorted_frames[i + 1]
        
        frame_links = linker.link(all_cells[frame1], all_cells[frame2])
        links.extend([
            (frame1, source, target, confidence)
            for source, target, confidence in frame_links
        ])
        linked_ids[frame1] = [(source, target) for source, target, _ in frame_links]
    
    # Analyze linking statistics
    print("Tracking Statistics:")
    print(f"Total frames: {len(all_cells)}")
    print(f"Total cells: {sum(len(cells) for cells in all_cells.values())}")
    print(f"Total links: {len(links)}")
    
    # Link quality analysis
    if links:
        confidences = [link[3] for link in links]
        print(f"\nLink Confidence:")
        print(f"  Mean: {np.mean(confidences):.3f}")
        print(f"  Std: {np.std(confidences):.3f}")
        print(f"  Min: {np.min(confidences):.3f}")
        print(f"  Max: {np.max(confidences):.3f}")
        
        # Plot confidence distribution
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(confidences, bins=30, edgecolor='black', alpha=0.7)
        ax.set_xlabel('Link Confidence Score')
        ax.set_ylabel('Frequency')
        ax.set_title('Distribution of Link Confidence Scores')
        ax.axvline(np.mean(confidences), color='r', linestyle='--', 
                    label=f'Mean: {np.mean(confidences):.3f}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()

## 3. Cell Division Analysis

In [ ]:
if linked_ids and all_cells:
    # Analyze cell fate at each frame
    cell_fates = defaultdict(lambda: {'appears': 0, 'continues': 0, 'divides': 0, 'disappears': 0})
    
    # Count links for each cell
    cell_parents = defaultdict(int)  # cell_id -> count of parents
    cell_children = defaultdict(int)  # cell_id -> count of children
    
    for frame, source, target, conf in links:
        cell_children[source] += 1
        cell_parents[target] += 1
    
    # Analyze fates
    all_cell_ids = set()
    for cells in all_cells.values():
        for cell in cells:
            cell_id = cell.get('id', cell.get('label', 0))
            all_cell_ids.add(cell_id)
    
    print("Cell Fate Analysis:")
    
    # Cells with multiple children (divisions)
    dividing_cells = {cid: count for cid, count in cell_children.items() if count > 1}
    print(f"Dividing cells (>1 child): {len(dividing_cells)}")
    
    # Cells with multiple parents (merges - likely errors)
    merging_cells = {cid: count for cid, count in cell_parents.items() if count > 1}
    print(f"Merging cells (>1 parent): {len(merging_cells)} (likely errors)")
    
    # Cells with no parents (appearances)
    appearing_cells = {cid for cid in all_cell_ids if cid not in cell_parents}
    print(f"Appearing cells (new tracks): {len(appearing_cells)}")
    
    # Cells with no children (disappearances)
    disappearing_cells = {cid for cid in all_cell_ids if cid not in cell_children}
    print(f"Disappearing cells (track ends): {len(disappearing_cells)}")

## 4. Error Analysis

In [ ]:
# Potential error detection
print("\nPotential Error Sources:")
print("="*50)

if all_cells:
    # Frame-by-frame analysis
    cell_counts = [len(cells) for cells in all_cells.values()]
    print(f"\nCell count variation:")
    print(f"  Min cells in frame: {np.min(cell_counts)}")
    print(f"  Max cells in frame: {np.max(cell_counts)}")
    print(f"  Std deviation: {np.std(cell_counts):.1f}")
    
    # Plot cell count over time
    fig, ax = plt.subplots(figsize=(12, 5))
    frames = sorted(all_cells.keys())
    cell_counts = [len(all_cells[f]) for f in frames]
    ax.plot(frames, cell_counts, 'b-o', linewidth=2, markersize=6)
    ax.fill_between(frames, cell_counts, alpha=0.3)
    ax.set_xlabel('Frame Index')
    ax.set_ylabel('Number of Cells')
    ax.set_title('Cell Count Over Time')
    ax.grid(True, alpha=0.3)
    plt.show()

print("\nCommon tracking errors:")
print("  - False negatives: Cells missed in segmentation")
print("  - False positives: Noise/artifacts detected as cells")
print("  - Identity switches: Same cell labeled as different IDs")
print("  - Fragmentation: Cell split into multiple segments")
print("  - Merging: Multiple cells linked into one track")

## 5. Summary and Recommendations

In [ ]:
print("Evaluation Summary:")
print("="*50)

if all_cells:
    print(f"\nDataset:")
    print(f"  Frames: {len(all_cells)}")
    print(f"  Total cells: {sum(len(cells) for cells in all_cells.values())}")
    print(f"  Average cells/frame: {np.mean([len(cells) for cells in all_cells.values()]):.1f}")

if links:
    print(f"\nTracking Performance:")
    print(f"  Total links: {len(links)}")
    print(f"  Link density: {len(links) / sum(len(cells) for cells in all_cells.values()):.2f}")
    print(f"  Mean confidence: {np.mean([link[3] for link in links]):.3f}")

print(f"\nRecommendations:")
print(f"  1. Optimize segmentation parameters for your data")
print(f"  2. Adjust max_distance threshold if needed")
print(f"  3. Review false positives and false negatives")
print(f"  4. Consider post-processing to remove short tracks")
print(f"\nNext step: Generate submission with final parameters")